In [ ]:
%%bash
set -e
if [ ! -d /content/UNITAS ]; then
  git clone https://github.com/DexForce/UNITAS.git /content/UNITAS
fi
ls /content/UNITAS


In [ ]:
%%bash
set -e
if [ ! -x /content/miniforge/bin/conda ]; then
  curl -fL https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -o /content/Miniforge3.sh
  bash /content/Miniforge3.sh -b -p /content/miniforge
fi
/content/miniforge/bin/conda --version


In [ ]:
%%bash
set -e
if [ ! -x /content/miniforge/envs/unitas/bin/python ]; then
  /content/miniforge/bin/conda create -n unitas python=3.10 pip -y
fi
/content/miniforge/bin/conda run --no-capture-output -n unitas python -m pip install torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
/content/miniforge/bin/conda run --no-capture-output -n unitas python -m pip install -e /content/UNITAS


In [ ]:
%%bash
set -e
cd /content/UNITAS
/content/miniforge/bin/conda run --no-capture-output -n unitas python -m pip install "huggingface_hub[cli]>=0.34,<1"
/content/miniforge/bin/conda run --no-capture-output -n unitas hf download microsoft/Florence-2-large \
  --local-dir checkpoints/Florence-2-large \
  --exclude "*.bin"
/content/miniforge/bin/conda run --no-capture-output -n unitas hf download RobbinWang123/Unitas unitas_scene_flow.pt \
  --local-dir checkpoints


In [ ]:
from pathlib import Path
import subprocess
import os
from google.colab import files

root = Path("/content/UNITAS")
script = root / "src/unitas/scene_flow/inference.py"
text = script.read_text()
old = 'torch.autocast(device.type, dtype=torch.bfloat16, enabled=device.type == "cuda")'
new = 'torch.autocast(device.type, dtype=torch.bfloat16, enabled=False)'
if old in text:
    assert text.count(old) == 1
    script.with_suffix(".py.original").write_text(text)
    script.write_text(text.replace(old, new))
else:
    assert new in text, "Unexpected inference.py contents"

subprocess.run([
    "/content/miniforge/envs/unitas/bin/python", "-u",
    "src/unitas/scene_flow/inference.py",
    "--checkpoint", "checkpoints/unitas_scene_flow.pt",
    "--florence-dir", "checkpoints/Florence-2-large",
    "--input", "assets/scene_flow/observations/libero_3d040e000e_0077.npz",
    "--output", "outputs/scene_flow.npz",
    "--device", "cuda",
    "--chunk-size", "256",
], cwd=root, check=True, env={**os.environ, "MPLBACKEND": "Agg", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2"})

files.download(str(root / "outputs/scene_flow.npz"))
